<a href="https://colab.research.google.com/github/arya7113/empathetic-response-ai/blob/main/empatheticResponseAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [ ]:
!pip install -q datasets --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.1 MB/s eta 0:00:00


In [ ]:
!pip install -q huggingface_hub transformers torch accelerate peft gradio scikit-learn

In [ ]:
from datasets import load_dataset

# Data Loading

In [ ]:
dataset = load_dataset("google-research-datasets/go_emotions", "simplified")
print(dataset)
print("\nFirst example:")
print(dataset["train"][0])

README.md:   0%|          | 0.00/9.40k [00:00<?, ?B/s]

simplified/train-00000-of-00001.parquet:   0%|          | 0.00/2.77M [00:00<?, ?B/s]

simplified/validation-00000-of-00001.par(…):   0%|          | 0.00/350k [00:00<?, ?B/s]

simplified/test-00000-of-00001.parquet:   0%|          | 0.00/347k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})

First example:
{'text': "My favourite food is anything I didn't have to cook myself.", 'labels': [27], 'id': 'eebbqej'}


# Feature Transformation

In [ ]:
LABELS = ["sadness", "joy", "anger", "fear", "surprise", "neutral"]
id2label = {i: l for i, l in enumerate(LABELS)}
label2id = {l: i for i, l in enumerate(LABELS)}

go_labels = dataset["train"].features["labels"].feature.names

MAPPING = {
    "sadness":"sadness","grief":"sadness","remorse":"sadness",
    "disappointment":"sadness","embarrassment":"sadness",

    "joy":"joy","amusement":"joy","excitement":"joy","gratitude":"joy",
    "love":"joy","optimism":"joy","pride":"joy","relief":"joy","admiration":"joy",

    "anger":"anger","annoyance":"anger","disapproval":"anger",

    "fear":"fear","nervousness":"fear",

    "surprise":"surprise","confusion":"surprise",
    "realization":"surprise","curiosity":"surprise",

    "neutral":"neutral","approval":"neutral","caring":"neutral",
    "desire":"neutral","disgust":"neutral",
}

In [ ]:
print(go_labels)

['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']


In [ ]:
print(id2label)

{0: 'sadness', 1: 'joy', 2: 'anger', 3: 'fear', 4: 'surprise', 5: 'neutral'}


In [ ]:
print(label2id)

{'sadness': 0, 'joy': 1, 'anger': 2, 'fear': 3, 'surprise': 4, 'neutral': 5}


In [ ]:
def encode_multilabel(example):
  vector = [0]*6
  for idx in example["labels"]:
    go_name = go_labels[idx]
    mapped = MAPPING.get(go_name, "neutral")
    vector[label2id[mapped]] = 1
    return {"labels": vector}

dataset = dataset.map(encode_multilabel)
print(dataset["train"][0])


Map:   0%|          | 0/43410 [00:00<?, ? examples/s]

Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

Map:   0%|          | 0/5427 [00:00<?, ? examples/s]

{'text': "My favourite food is anything I didn't have to cook myself.", 'labels': [0, 0, 0, 0, 0, 1], 'id': 'eebbqej'}


# Tokenization

In [ ]:
from transformers import AutoTokenizer
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
  return tokenizer(
      batch["text"],
      padding="max_length",
      truncation=True,
      max_length=128,
  )
tokenized = dataset.map(tokenize, batched=True)

class CustomDataCollator:
    def __call__(self, batch):
        input_ids = torch.stack([torch.tensor(x["input_ids"]) for x in batch])
        attention_mask = torch.stack([torch.tensor(x["attention_mask"]) for x in batch])
        labels = torch.stack([torch.tensor(x["labels"], dtype=torch.float32) for x in batch])

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

# Don't use set_format — use the custom collator instead
collator = CustomDataCollator()

# Model Building

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=6,
    id2label=id2label,
    label2id=label2id,
    problem_type="multi_label_classification"   # tells model to expect vectors
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# # Sanity check — confirms multi-label loss works WITHOUT training anything yet
# batch = {k: v[:4].to(model.device) for k, v in tokenized["train"][:4].items()
#          if k in ["input_ids","attention_mask","labels"]}
# batch["labels"] = batch["labels"].float()
# out = model(**batch)
# print(out.loss)

tensor(0.7210, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)


In [ ]:
import numpy as np
from sklearn.metrics import f1_score
import torch

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Apply sigmoid and threshold at 0.5
    preds = (torch.sigmoid(torch.tensor(logits)) > 0.5).int().numpy()
    # labels is already float, convert to int for f1_score
    labels = (torch.tensor(labels) > 0.5).int().numpy()
    f1 = f1_score(labels, preds, average="weighted", zero_division=0)
    return {"f1": f1}

# Model Training

In [ ]:
# from transformers import TrainingArguments, Trainer

# args = TrainingArguments(
#     output_dir="./emotion-classifier",
#     num_train_epochs=3,
#     per_device_train_batch_size=32,
#     per_device_eval_batch_size=64,
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     load_best_model_at_end=True,
#     logging_steps=100,
#     fp16=True,
# )

# trainer = Trainer(
#     model=model,
#     args=args,
#     train_dataset=tokenized["train"],
#     eval_dataset=tokenized["validation"],
#     data_collator=collator,
#     compute_metrics=compute_metrics,
# )

# trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,0.231769,0.213674,0.709924
2,0.187101,0.219585,0.711618
3,0.126243,0.242142,0.713477


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4071, training_loss=0.190295875584924, metrics={'train_runtime': 422.1147, 'train_samples_per_second': 308.518, 'train_steps_per_second': 9.644, 'total_flos': 4313114982927360.0, 'train_loss': 0.190295875584924, 'epoch': 3.0})

In [ ]:
def detect_emotions(text, threshold=0.5):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.sigmoid(logits[0])
    detected = [
        {"emotion": LABELS[i], "score": round(p.item(), 3)}
        for i, p in enumerate(probs) if p.item() > threshold
    ]
    return sorted(detected, key=lambda x: -x["score"])

print(detect_emotions("I finally got the job but I'm terrified"))
print(detect_emotions("I failed my exam again"))
print(detect_emotions("My boss keeps ignoring me, I'm done"))

[{'emotion': 'fear', 'score': 0.846}]
[{'emotion': 'sadness', 'score': 0.639}]
[{'emotion': 'anger', 'score': 0.572}]


# Model Saving

In [ ]:
model.save_pretrained("./emotion-classifier")
tokenizer.save_pretrained("./emotion-classifier")
print("Saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved!


In [ ]:
model.save_pretrained(
    "/content/drive/MyDrive/emotion-classifier"
)

tokenizer.save_pretrained(
    "/content/drive/MyDrive/emotion-classifier"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/emotion-classifier/tokenizer_config.json',
 '/content/drive/MyDrive/emotion-classifier/tokenizer.json')

#*----------------------------------PHASE - 2---------------------------------------*

# Model Loading

In [ ]:
!pip install "datasets<4.0.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 11.7 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [3]:
!pip uninstall torchvision -y -q
!pip install transformers peft accelerate -q
!pip install torchao --upgrade -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 106.2 MB/s eta 0:00:00


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

classifier_model = AutoModelForSequenceClassification.from_pretrained(
    "/content/drive/MyDrive/emotion-classifier"
)
classifier_tokenizer = AutoTokenizer.from_pretrained(
    "/content/drive/MyDrive/emotion-classifier"
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

# Data Loading

In [ ]:
# from datasets import load_dataset

# ds2 = load_dataset("facebook/empathetic_dialogues")

In [ ]:
# from datasets import load_dataset

# ds = load_dataset("Estwld/empathetic_dialogues_llm")
# print(ds["train"][0])

In [ ]:
# ds = load_dataset("nbertagnolli/counsel-chat")
# print(ds)
# print(ds["train"][0])

In [11]:
import pandas as pd
from datasets import Dataset

# Load CSV
df = pd.read_csv("/content/drive/MyDrive/empathy_dataset .csv")

# Quick sanity check
print(df.shape)
print(df.columns.tolist())
print(df.head(3))
print(df["emotion"].value_counts())

(4550, 3)
['emotion', 'context', 'response']
   emotion                                            context  \
0  sadness  got my calc midterm back. 54. studied for two ...   
1  sadness  my dad saw the 54 on the portal before i even ...   
2  sadness  bestie got a 96 on the same calc test. bus rid...   

                                            response  
0  Two weeks of studying and the number still cam...  
1  Having the portal tell your dad before you cou...  
2  Sitting in that quiet bus next to a 96 while y...  
emotion
sadness     759
surprise    759
joy         758
anger       758
fear        758
neutral     758
Name: count, dtype: int64


In [12]:
question_count = df['response'].str.contains(r'\?', regex=True).sum()
print(f"{question_count} / {len(df)} responses contain a '?'")
print(df[df['response'].str.contains(r'\?', regex=True)]['response'].head(10).tolist())

2550 / 4550 responses contain a '?'
["Two weeks of studying and the number still came back a 54 — that gap between effort and result is its own special ache. One bad midterm doesn't erase the fact that you showed up for it every day. What felt different between practicing at home and sitting in the actual test?", 'Having the portal tell your dad before you could is such a specific kind of powerless — the story got out without you in it. You deserved the chance to frame your own bad news. How did you want that conversation to go, in a perfect world?', "Sitting in that quiet bus next to a 96 while you're holding your own score is a lonely little eternity. Being happy for her and sad for you can both be true at once. What would you want her to say, if she knew?", "No wonder your stomach dropped — 'see me after class' lands like a verdict when the whole room hears it. The public part stings separately from the midterm itself. Which one is sitting heavier tonight, the grade or the audience?

In [13]:
import re

def strip_trailing_question(response):
    # Split into sentences, drop trailing ones that are questions
    sentences = re.split(r'(?<=[.!?])\s+', response.strip())
    while sentences and sentences[-1].strip().endswith('?'):
        sentences.pop()
    return ' '.join(sentences).strip()


df['response_clean'] = df['response'].apply(strip_trailing_question)
df_final = df[['emotion', 'context', 'response_clean']].rename(columns={'response_clean': 'response'})
# df_final.to_csv("empathy_dataset_v2.csv", index=False)

In [ ]:
# Convert to HuggingFace Dataset
ds = Dataset.from_pandas(df_final)

# Format for GPT-2 training
def format_prompt(example):
    return {
        "text": f"[Emotion: {example['emotion']}] User: {example['context']}\nResponse: {example['response']}<|endoftext|>"
    }

ds_formatted = ds.map(format_prompt)
print(ds_formatted[0])

Map:   0%|          | 0/4550 [00:00<?, ? examples/s]

{'emotion': 'sadness', 'context': 'got my calc midterm back. 54. studied for two weeks straight lol', 'response': "Two weeks of studying and the number still came back a 54 — that gap between effort and result is its own special ache. One bad midterm doesn't erase the fact that you showed up for it every day.", 'text': "[Emotion: sadness] User: got my calc midterm back. 54. studied for two weeks straight lol\nResponse: Two weeks of studying and the number still came back a 54 — that gap between effort and result is its own special ache. One bad midterm doesn't erase the fact that you showed up for it every day.<|endoftext|>"}


## Dataset Cleaning

In [ ]:
# from transformers import AutoTokenizer

# gen_tokenizer = AutoTokenizer.from_pretrained("gpt2")
# gen_tokenizer.pad_token = gen_tokenizer.eos_token  # GPT-2 has no pad token by default

# def tokenize(batch):
#     return gen_tokenizer(
#         batch["text"],
#         max_length=128,
#         truncation=True,
#         padding="max_length"
#     )

# tokenized = ds_formatted.map(tokenize, batched=True)
# tokenized.set_format("torch", columns=["input_ids", "attention_mask"])
# print(tokenized)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

dialo_tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-medium")
dialo_tokenizer.pad_token = dialo_tokenizer.eos_token

# gen_tokenizer = AutoTokenizer.from_pretrained("gpt2")
# gen_tokenizer.pad_token = gen_tokenizer.eos_token  # GPT-2 has no pad token by default



config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

In [ ]:
def tokenize_and_mask_dialo(examples):
    input_ids_list, attention_mask_list, labels_list = [], [], []

    for text in examples["text"]:
        full = dialo_tokenizer(text, max_length=256, truncation=True, padding="max_length")
        input_ids = full["input_ids"]
        attn = full["attention_mask"]

        prompt_text = text.split("Response:")[0] + "Response:"
        prompt_ids = dialo_tokenizer(prompt_text, truncation=True, max_length=256)["input_ids"]
        prompt_len = len(prompt_ids)

        labels = input_ids.copy()
        labels[:prompt_len] = [-100] * prompt_len
        labels = [l if a == 1 else -100 for l, a in zip(labels, attn)]

        input_ids_list.append(input_ids)
        attention_mask_list.append(attn)
        labels_list.append(labels)

    return {"input_ids": input_ids_list, "attention_mask": attention_mask_list, "labels": labels_list}

tokenized = ds_formatted.map(tokenize_and_mask_dialo, batched=True, remove_columns=ds_formatted.column_names)
split = tokenized.train_test_split(test_size=0.1, seed=42)
split = split.with_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/4550 [00:00<?, ? examples/s]

In [ ]:
print(type(split["train"][0]["input_ids"]))
# Should print: <class 'torch.Tensor'>

<class 'torch.Tensor'>


In [ ]:
print(ds_formatted[0])
print(split)

{'emotion': 'sadness', 'context': 'got my calc midterm back. 54. studied for two weeks straight lol', 'response': "Two weeks of studying and the number still came back a 54 — that gap between effort and result is its own special ache. One bad midterm doesn't erase the fact that you showed up for it every day.", 'text': "[Emotion: sadness] User: got my calc midterm back. 54. studied for two weeks straight lol\nResponse: Two weeks of studying and the number still came back a 54 — that gap between effort and result is its own special ache. One bad midterm doesn't erase the fact that you showed up for it every day.<|endoftext|>"}
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 4095
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 455
    })
})


In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    task_type="CAUSAL_LM",
    lora_dropout=0.1,
    target_modules=["c_attn", "c_proj"],
    bias="none"
)

base_model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-medium")
# base_model = AutoModelForCausalLM.from_pretrained("gpt2")

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  863MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  863MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


trainable params: 2,162,688 || all params: 356,985,856 || trainable%: 0.6058


In [ ]:
from transformers import TrainingArguments, Trainer, default_data_collator

data_collator = default_data_collator


args = TrainingArguments(
    output_dir="./empathy-lora",
    num_train_epochs=10,              # only change this
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50,
    fp16=True,
    warmup_steps=100,
    learning_rate=2e-4,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    data_collator=data_collator,
)

trainer.train()

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,4.232390,3.462313
2,3.400471,3.159799
3,3.273494,3.050605
4,3.157085,2.987943
5,3.096757,2.948298
6,3.075199,2.922983
7,3.011197,2.906331
8,3.007009,2.898073
9,2.952263,2.889650
10,2.960378,2.885509


TrainOutput(global_step=1280, training_loss=3.2943343579769135, metrics={'train_runtime': 2690.4671, 'train_samples_per_second': 15.22, 'train_steps_per_second': 0.476, 'total_flos': 1.91511780655104e+16, 'train_loss': 3.2943343579769135, 'epoch': 10.0})

In [ ]:
import os

save_path = "/content/drive/MyDrive/empathy-response-generator"
os.makedirs(save_path, exist_ok=True)

# Saves only the LoRA adapter weights (small, a few MB) — not the full 345M base model
model.save_pretrained(save_path)
dialo_tokenizer.save_pretrained(save_path)

print(f"Saved to {save_path}")
print(os.listdir(save_path))

Saved to /content/drive/MyDrive/empathy-response-generator
['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json']


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

save_path = "/content/drive/MyDrive/empathy-response-generator"  # wherever you saved it

reload_tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-medium")
reload_tokenizer.pad_token = reload_tokenizer.eos_token

reload_base = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-medium")
reload_model = PeftModel.from_pretrained(reload_base, save_path)
reload_model.eval()

print("Reloaded successfully")

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

Reloaded successfully


In [7]:
print(reload_model.device)

cuda:0


In [6]:
reload_model = reload_model.to("cuda")

In [8]:
import torch
import re


def generate_candidates(user_text, emotion, n_candidates=5, max_new_tokens=40):
    prompt = f"[Emotion: {emotion}] User: {user_text}\nResponse:"
    inputs = reload_tokenizer(prompt, return_tensors="pt").to(reload_model.device)

    with torch.no_grad():
        outputs = reload_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.5,
            do_sample=True,
            top_p=0.85,
            repetition_penalty=1.15,
            no_repeat_ngram_size=3,
            pad_token_id=reload_tokenizer.eos_token_id,
            eos_token_id=reload_tokenizer.eos_token_id,
            num_return_sequences=n_candidates,   # generates all candidates in one batched call
        )

    candidates = []
    for output in outputs:
        generated = reload_tokenizer.decode(output, skip_special_tokens=True)
        response = generated.split("Response:")[-1].strip()
        candidates.append(response)

    return candidates


def score_candidate(response, user_text):
    """
    Simple heuristic grounding score — penalizes likely hallucination signals.
    Not perfect, but catches the most common failure patterns we've seen.
    """
    score = 100

    # Penalize trailing questions (shouldn't happen post-cleaning, but double-check)
    if response.strip().endswith("?"):
        score -= 30

    # Penalize third-person pronouns not implied by context
    # (crude heuristic: flag "he", "she", "him", "her", "they" appearing in response
    # but not in the original user_text)
    pronouns = ["he ", "she ", "him ", "her ", "they ", "them "]
    user_lower = user_text.lower()
    for pronoun in pronouns:
        if pronoun in response.lower() and pronoun not in user_lower:
            score -= 25

    # Penalize very short or very long responses (likely truncated or rambling)
    word_count = len(response.split())
    if word_count < 5:
        score -= 40
    elif word_count > 45:
        score -= 15

    # Penalize obvious repetition (same word 3+ times)
    words = response.lower().split()
    for word in set(words):
        if len(word) > 3 and words.count(word) >= 3:
            score -= 20

    # Penalize garbled artifacts (underscores, double spaces, broken tokens)
    if "_" in response or "  " in response:
        score -= 20

    return score


def generate_best_response(user_text, emotion, n_candidates=5, verbose=False):
    """Generate n candidates, score them, return the best one."""
    candidates = generate_candidates(user_text, emotion, n_candidates=n_candidates)
    scored = [(c, score_candidate(c, user_text)) for c in candidates]
    scored.sort(key=lambda x: x[1], reverse=True)

    if verbose:
        for c, s in scored:
            print(f"  [{s}] {c}")

    return scored[0][0]  # return highest-scoring candidate


# --- Test it ---
test_cases = [
    ("I failed my exam again", "sadness"),
    ("I finally got the job!", "joy"),
    ("My boss keeps ignoring me", "anger"),
    ("I have a biopsy scheduled next week and I can't stop thinking about it", "fear"),
]

for text, emotion in test_cases:
    print(f"[{emotion.upper()}] {text}")
    best = generate_best_response(text, emotion, n_candidates=5, verbose=True)
    print(f"  → BEST: {best}\n")

[SADNESS] I failed my exam again


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


  [100] Failing your exam twice is a quiet, heavy loss. Grieving that hurt is completely understandable.
  [75] Failing to pass the first time is a real, specific grief. That ache is completely valid, and you're allowed to feel it.
  [75] Failing the one you trusted most is a real, heavy grief. Grieving that hurt is completely valid.
  [75] Failing a test again is a real, real grief. Grieving that you didn't get the chance to finish it is completely understandable.
  [50] Reaching the point where you're not even sure whether to celebrate or mourn. That loss is a real, specific grief.
  → BEST: Failing your exam twice is a quiet, heavy loss. Grieving that hurt is completely understandable.

[JOY] I finally got the job!
  [100] Getting that first job is a real milestone, and it's worth celebrating. Being proud of yourself is worth celebrating too.
  [100] Getting a little bit of that pride back is worth celebrating. Being proud of yourself again is worth every second.
  [100] That's a mi

In [ ]:
import torch

def generate_response_dialo(user_text, emotion, max_new_tokens=40):
    prompt = f"[Emotion: {emotion}] User: {user_text}\nResponse:"
    inputs = dialo_tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.8,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            no_repeat_ngram_size=3,
            pad_token_id=dialo_tokenizer.eos_token_id,
            eos_token_id=dialo_tokenizer.eos_token_id,
        )

    generated = dialo_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated.split("Response:")[-1].strip()

test_cases = [
    # Sadness
    ("I just found out my childhood home was sold to strangers", "sadness"),
    ("My best friend moved to another country and we barely talk now", "sadness"),

    # Joy
    ("I finally paid off my student loans after 6 years", "joy"),
    ("My garden bloomed for the first time this spring", "joy"),

    # Anger
    ("Someone took credit for my project in front of the whole team", "anger"),
    ("My landlord raised the rent without any notice", "anger"),

    # Fear
    ("I have a biopsy scheduled next week and I can't stop thinking about it", "fear"),
    ("I'm about to give a presentation to the entire company tomorrow", "fear"),

    # Surprise
    ("My coworkers threw me a surprise party I had no idea about", "surprise"),
    ("I just found out I'm getting a sibling after being an only child for 25 years", "surprise"),

    # Neutral
    ("I switched my morning coffee brand and it's fine, nothing special", "neutral"),
    ("I finished reorganizing my closet today", "neutral"),
]

for text, emotion in test_cases:
    print(f"[{emotion.upper()}] {text}")
    print(f"  → {generate_response_dialo(text, emotion)}")
    print()

[SADNESS] I just found out my childhood home was sold to strangers
  → Mourning that whole family is a quiet grief. Grieving that connection in such a small way is completely understandable.

[SADNESS] My best friend moved to another country and we barely talk now
  → Having your friends you love moved away in the middle of the conversation is a lonely, lonely place. Grieving that they left is completely fair.

[JOY] I finally paid off my student loans after 6 years
  → That much cash finally landed you is a real milestone. It makes real, long-term sense.

[JOY] My garden bloomed for the first time this spring
  → A spring that bright and full of flowers is the first real sight of a whole life you built. That bloom is proof you're living in the right place at once.

[ANGER] Someone took credit for my project in front of the whole team
  → Being handed a piece of work for something you built, and then being insulted by it is a genuinely maddening thing. Your anger at that is completely 